# Projet : Classification d'images de fruits et légumes

Bienvenue dans ce notebook ! Dans ce projet, nous allons construire un réseau de neurones convolutifs (CNN) de A à Z pour la classification d’images. Même si les modèles préentraînés offrent souvent une meilleure précision, cette approche n’est pas toujours la plus adaptée. Ici, l’objectif est de créer un CNN entièrement conçu depuis le début, sans utiliser de poids préentraînés.

Usage métier : ce dataset a été choisi pour ses applications possibles en entreprise. Un modèle de classification de fruits et légumes peut être intégré à des applications mobiles pour identifier automatiquement les produits et proposer des recettes. Il peut aussi servir dans des magasins automatisés comme Amazon Go/dans les cantines d'entreprise pour reconnaître les articles pris par les clients et faciliter un passage en caisse sans contact.

## Description du dataset

**Overview :**  
Ce dataset contient des images de différents fruits et légumes, destinées à un problème de classification d’images.

- **Fruits :** Banana, Apple, Pear, Grapes, Orange, Kiwi, Watermelon, Pomegranate, Pineapple, Mango
- **Légumes :** Cucumber, Carrot, Capsicum, Onion, Potato, Lemon, Tomato, Radish, Beetroot, Cabbage, Lettuce, Spinach, Soybean, Cauliflower, Bell Pepper, Chilli Pepper, Turnip, Corn, Sweetcorn, Sweet Potato, Paprika, Jalapeño, Ginger, Garlic, Peas, Eggplant

**Données :**
- **Train Set :** 100 images par catégorie
- **Test Set :** 10 images par catégorie
- **Validation Set :** 10 images par catégorie

Les images sont organisées en sous-dossiers par classe, pour chaque ensemble : `train`, `test`, `validation`.

**Source des données :**  
Les images ont été collectées via Bing Image Search pour ce projet personnel.



### Consignes :
1. Remplissez les sections TODO dans les cellules Python.
2. Conservez la structure du notebook et exécutez les cellules dans l'ordre.
3. Ajoutez des commentaires lorsque vous modifiez ou améliorez le code.
4. Utilisez les astuces TIP pour guider votre travail.
5. Documentez les résultats et les observations dans les cellules Markdown.

# Installation des librairies

In [ ]:
# Installer les packages nécessaires
# !pip install matplotlib
# !pip install seaborn
# !pip install scikit-learn
# !pip install tensorflow
# !pip install visualkeras

# 1. Importation des librairies

In [ ]:
import os
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.cm as cm

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
# Pour TensorFlow DirectML: 0 = premier adaptateur détecté (ici NVIDIA)
os.environ["DML_VISIBLE_DEVICES"] = "0"

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint

from PIL import Image
import visualkeras
import pickle
import warnings


# Ignore all warnings
warnings.filterwarnings("ignore")

# Vérifier si TensorFlow voit un GPU et, si oui, activer la croissance mémoire.
# Sur Windows natif avec TensorFlow >= 2.11, le GPU CUDA n'est pas supporté.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print('GPU détecté par TensorFlow :')
    for gpu in gpus:
        print(f'- {gpu}')
else:
    print('Aucun GPU TensorFlow détecté dans ce runtime.')
    print("Sous Windows natif, TensorFlow 2.11+ n'utilise pas CUDA; pour du GPU il faut WSL2 ou un backend compatible GPU.")

# 2. Data Visualization

### 2.1 Dataset Sample

In [ ]:
# TODO: Afficher un échantillon d'images du dossier d'entraînement
# TIP: Utilisez matplotlib pour afficher une grille d'images et vérifier les tailles et couleurs.
# Exemple:
# vis_dir = 'fruit-and-vegetable-image-recognition/train'
# ...

vis_dir = 'fruit-and-vegetable-image-recognition/train'

classes = sorted([d for d in os.listdir(vis_dir) if os.path.isdir(os.path.join(vis_dir, d))])
sample_paths = []

for cls in classes:
    cls_dir = os.path.join(vis_dir, cls)
    images = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if images:
        sample_paths.append((cls, os.path.join(cls_dir, random.choice(images))))

n_cols = 6
n_rows = int(np.ceil(len(sample_paths) / n_cols))

plt.figure(figsize=(18, 3 * n_rows))
for idx, (cls_name, img_path) in enumerate(sample_paths, start=1):
    plt.subplot(n_rows, n_cols, idx)
    img = mpimg.imread(img_path)
    plt.imshow(img)
    plt.title(cls_name, fontsize=9)
    plt.axis('off')

plt.suptitle('Echantillon d\'images du train set', fontsize=16)
plt.tight_layout()
plt.show()

### 2.2 Training Data Distribution

In [ ]:

# TODO: Compter le nombre d'images par classe dans l'ensemble d'entraînement
# TIP: Un dataset bien équilibré permet d'éviter les biais lors de l'entraînement.
# Exemple:
# classes = os.listdir(vis_dir)
# image_count = {cls: len(os.listdir(os.path.join(vis_dir, cls))) for cls in classes}
# ...

base_dir = 'fruit-and-vegetable-image-recognition'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')
validation_dir = os.path.join(base_dir, 'validation')

def count_images_per_class(directory):
    class_counts = {}
    classes_local = sorted([d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))])
    for cls in classes_local:
        cls_dir = os.path.join(directory, cls)
        image_files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        class_counts[cls] = len(image_files)
    return class_counts

def plot_distribution(class_counts, title):
    labels = list(class_counts.keys())
    values = list(class_counts.values())

    plt.figure(figsize=(14, 6))
    sns.barplot(x=labels, y=values, color='steelblue')
    plt.xticks(rotation=90)
    plt.ylabel('Nombre d\'images')
    plt.title(title)
    plt.tight_layout()
    plt.show()

train_counts = count_images_per_class(train_dir)
print(f'Nombre de classes (train): {len(train_counts)}')
print(f'Exemple de classes: {list(train_counts.keys())[:5]}')
plot_distribution(train_counts, 'Distribution des classes - Train Set')

### 2.3 Test Data Distribution

In [ ]:
# TODO: Vérifier la distribution des images dans l'ensemble de test
# TIP: Utilisez le même code que pour l'ensemble d'entraînement en changeant le chemin.

test_counts = count_images_per_class(test_dir)
print(f'Nombre de classes (test): {len(test_counts)}')
plot_distribution(test_counts, 'Distribution des classes - Test Set')

### 2.4 Validation Data Distribution

In [ ]:
# TODO: Vérifier la distribution des images dans l'ensemble de validation
# TIP: Le dataset de validation doit être équilibré pour obtenir une évaluation fiable.

validation_counts = count_images_per_class(validation_dir)
print(f'Nombre de classes (validation): {len(validation_counts)}')
plot_distribution(validation_counts, 'Distribution des classes - Validation Set')

# 3. Resizing Images

### 3.1 Resizing Images to 128x128 Pixels

In [ ]:

# TODO: Redimensionner les images pour créer un dataset uniformisé
# TIP: Convertissez les images en RGB et enregistrez-les dans un nouveau dossier `resized_images`.
# Exemple:
# def resize_and_save_image(input_path, output_path, size=(128, 128)):
#     ...
# def process_directory(input_directory, output_directory):
#     ...

resized_root = 'resized'
target_size = (128, 128)

def resize_and_save_image(input_path, output_path, size=(128, 128)):
    try:
        with Image.open(input_path) as img:
            img = img.convert('RGB')
            img = img.resize(size)
            img.save(output_path, quality=95)
    except Exception as e:
        print(f'Erreur lors du traitement de {input_path}: {e}')
        return False
    return True

def process_directory(input_directory, output_directory, size=(128, 128)):
    os.makedirs(output_directory, exist_ok=True)
    classes_local = sorted([d for d in os.listdir(input_directory) if os.path.isdir(os.path.join(input_directory, d))])

    for cls in classes_local:
        src_cls_dir = os.path.join(input_directory, cls)
        dst_cls_dir = os.path.join(output_directory, cls)
        os.makedirs(dst_cls_dir, exist_ok=True)

        for file_name in os.listdir(src_cls_dir):
            src_path = os.path.join(src_cls_dir, file_name)
            # Ignorer les éléments qui ne sont pas des fichiers ordinaires
            if not os.path.isfile(src_path):
                continue
            # Ignorer les fichiers qui ne semblent pas être des images
            if not file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue

            dst_path = os.path.join(dst_cls_dir, file_name)
            success = resize_and_save_image(src_path, dst_path, size=size)
            if not success:
                if os.path.exists(dst_path):
                    try:
                        os.remove(dst_path)
                    except Exception:
                        pass

for subset in ['train', 'test', 'validation']:
    src_dir = os.path.join(base_dir, subset)
    dst_dir = os.path.join(resized_root, subset)
    if not os.path.isdir(src_dir):
        print(f'Source introuvable, skipping {src_dir}')
        continue
    process_directory(src_dir, dst_dir, size=target_size)
    print(f'Redimensionnement terminé pour {subset}: {dst_dir}')

### 3.2 Resized Images Visualization

In [ ]:
# TODO: Afficher un échantillon d'images redimensionnées
# TIP: Vérifiez que toutes les images ont bien la même taille et qu'elles restent lisibles.

resized_train_dir = os.path.join(resized_root, 'train')
resized_classes = sorted([d for d in os.listdir(resized_train_dir) if os.path.isdir(os.path.join(resized_train_dir, d))])

resized_samples = []
for cls in resized_classes:
    cls_dir = os.path.join(resized_train_dir, cls)
    images = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if images:
        resized_samples.append((cls, os.path.join(cls_dir, random.choice(images))))

n_cols = 6
n_rows = int(np.ceil(len(resized_samples) / n_cols))

plt.figure(figsize=(18, 3 * n_rows))
for idx, (cls_name, img_path) in enumerate(resized_samples, start=1):
    plt.subplot(n_rows, n_cols, idx)
    img = mpimg.imread(img_path)
    plt.imshow(img)
    plt.title(f'{cls_name}\n{img.shape[1]}x{img.shape[0]}', fontsize=8)
    plt.axis('off')

plt.suptitle('Echantillon d\'images redimensionnees (128x128)', fontsize=16)
plt.tight_layout()
plt.show()

# 4. Data Generators

In [ ]:
# TODO: Créer les générateurs d'images pour l'entraînement et le test
# TIP: Utilisez ImageDataGenerator avec `rescale=1./255` et éventuellement quelques augmentations légères.
# Exemple:
# train_datagen = ImageDataGenerator(rescale=1./255, ...)
# test_datagen = ImageDataGenerator(rescale=1./255)
# train_generator = train_datagen.flow_from_directory(...)
# test_generator = test_datagen.flow_from_directory(...)

IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.05,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode='nearest'
    )

eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(resized_root, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
    )

test_generator = eval_datagen.flow_from_directory(
    os.path.join(resized_root, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
    )

validation_generator = eval_datagen.flow_from_directory(
    os.path.join(resized_root, 'validation'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
    )

num_classes = len(train_generator.class_indices)
print(f'Nombre de classes detectees: {num_classes}')

# 5. CNN Creation

In [ ]:
# TODO: Définir l'architecture du CNN
# TIP: 5 couches Conv2D + MaxPooling, puis ajoutez des BatchNormalization et un flatten suivi de deux Dense puis la sortie Dense.
# Exemple:
# model = models.Sequential([
#     layers.Conv2D(...),
#     ...
#     layers.Dense(36, activation='softmax')
# ])
# model.compile(...)
# model.summary()

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
    )

model.summary()


# 6. Model Training

In [ ]:
# TODO: Entraîner le modèle avec validation
# TIP: Utilisez ModelCheckpoint pour sauvegarder le meilleur modèle selon `val_accuracy`. Prendre 150 epochs
# Exemple:
# checkpoint_callback = ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
# history = model.fit(train_generator, epochs=..., validation_data=test_generator, callbacks=[checkpoint_callback])
# with open('training_history.pkl', 'wb') as f: pickle.dump(history.history, f)

checkpoint_callback = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
    )

history = model.fit(
    train_generator,
    epochs=150,
    validation_data=validation_generator,
    callbacks=[checkpoint_callback]
    )

with open('training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

print('Historique sauvegardé dans training_history.pkl')

# 7. CNN Visualization

In [ ]:
# TODO: Visualiser l'architecture du meilleur modèle
# TIP: Utilisez VisualKeras ou imprimez simplement le résumé du modèle.
# Exemple:
# visualkeras.layered_view(model).save('model_visualization.png')

import io
import contextlib

best_model = load_model('best_model.keras')
best_model.summary()

try:
    visualkeras.layered_view(best_model, legend=True).save('model_visualization.png')
    print('Visualisation sauvegardee dans model_visualization.png')
except Exception as visual_error:
    print(f'VisualKeras indisponible ou incompatible: {visual_error}')
    try:
        from tensorflow.keras.utils import plot_model
        plot_model(best_model, to_file='model_visualization.png', show_shapes=True, show_layer_names=True, dpi=120)
        print('Visualisation sauvegardee avec plot_model dans model_visualization.png')
    except Exception as plot_error:
        summary_buffer = io.StringIO()
        with contextlib.redirect_stdout(summary_buffer):
            best_model.summary()
        summary_text = summary_buffer.getvalue()

        plt.figure(figsize=(14, max(6, len(summary_text.splitlines()) * 0.35)))
        plt.axis('off')
        plt.text(0.01, 0.99, summary_text, va='top', ha='left', family='monospace', fontsize=10)
        plt.tight_layout()
        plt.savefig('model_visualization.png', dpi=200, bbox_inches='tight')
        plt.close()
        print(f'Fallback texte sauvegardé dans model_visualization.png: {plot_error}')

# 8. Training Results

### 8.1 Training Loss & Accuracy

In [ ]:
# TODO: Afficher les courbes de loss et d'accuracy
# TIP: Chargez l'historique depuis le fichier pickle et tracez les courbes d'entraînement et de validation.

with open('training_history.pkl', 'rb') as f:
    history_data = pickle.load(f)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history_data['loss'], label='Train Loss')
plt.plot(history_data['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_data['accuracy'], label='Train Accuracy')
plt.plot(history_data['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

### 8.2 Model Confusion Matrix

In [ ]:
# TODO: Calculer et afficher la matrice de confusion
# TIP: Utilisez `model.predict` sur le générateur de test, puis `confusion_matrix(labels, predictions, normalize='true')`.

best_model = load_model('best_model.keras')

test_generator.reset()
pred_probs = best_model.predict(test_generator, verbose=1)
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = test_generator.classes

cm = confusion_matrix(true_labels, pred_labels, normalize='true')

class_labels = [None] * len(test_generator.class_indices)
for label, idx in test_generator.class_indices.items():
    class_labels[idx] = label

plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
plt.title('Matrice de confusion normalisee - Test Set')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

### 8.3 Model Classification Report

In [ ]:
# TODO: Générer le rapport de classification
# TIP: Utilisez `classification_report` avec les noms de classes pour obtenir precision, recall et f1-score.

print('Accuracy (test):', accuracy_score(true_labels, pred_labels))
print()
print(classification_report(true_labels, pred_labels, target_names=class_labels))

# 9. Testing On Validation Data

### 9.1 Predictions On Validation Images

In [ ]:
# TODO: Charger des images de validation individuelles et faire des prédictions

# Load the trained model
best_model = load_model('best_model.keras')

# Define paths to the images
image_paths = [
    'fruit-and-vegetable-image-recognition/validation/apple/Image_1.jpg',
    'fruit-and-vegetable-image-recognition/validation/banana/Image_2.jpg',
    'fruit-and-vegetable-image-recognition/validation/orange/Image_3.jpg'
]
# TIP: Pré-traitez les images de la même façon que pour le générateur.
# Exemple:
# img = image.load_img(path, target_size=(128, 128))
# img_array = image.img_to_array(img) / 255.0
# pred = model.predict(np.expand_dims(img_array, axis=0))

class_names = [None] * len(train_generator.class_indices)
for label, idx in train_generator.class_indices.items():
    class_names[idx] = label

def preprocess_for_prediction(img_path, target_size=(128, 128)):
    img = image.load_img(img_path, target_size=target_size)
    img_array = image.img_to_array(img) / 255.0
    return np.expand_dims(img_array, axis=0), img

for path in image_paths:
    if not os.path.exists(path):
        print(f'Image introuvable: {path}')
        continue

    img_batch, pil_img = preprocess_for_prediction(path)
    probs = best_model.predict(img_batch, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = class_names[pred_idx]
    confidence = float(probs[pred_idx])

    plt.figure(figsize=(4, 4))
    plt.imshow(pil_img)
    plt.title(f'Prediction: {pred_label} ({confidence:.2%})')
    plt.axis('off')
    plt.show()

### 9.2 Validation Data Classification Report

In [ ]:
# TODO: Évaluer le modèle sur tout l'ensemble de validation
# TIP: Créez un `val_generator` avec `shuffle=False` pour conserver l'ordre des labels.

val_datagen = ImageDataGenerator(rescale=1.0 / 255)
val_generator = val_datagen.flow_from_directory(
    os.path.join(resized_root, 'validation'),
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
    )

best_model = load_model('best_model.keras')
val_generator.reset()
val_probs = best_model.predict(val_generator, verbose=1)
val_pred = np.argmax(val_probs, axis=1)
val_true = val_generator.classes

val_class_names = [None] * len(val_generator.class_indices)
for label, idx in val_generator.class_indices.items():
    val_class_names[idx] = label

print('Validation Accuracy:', accuracy_score(val_true, val_pred))
print()
print(classification_report(val_true, val_pred, target_names=val_class_names))

# 10. Saliency Maps for CNN Explainability

In [ ]:
# TODO: Calculer et afficher des cartes de saillance

# Load the best model
best_model = load_model('best_model.keras')

# List of image paths
img_paths = [
    'resized/train/chilli pepper/Image_36.jpg',
    'resized/train/tomato/Image_9.jpg',
    'resized/train/lemon/Image_9.jpg',
    'resized/train/onion/Image_26.jpg'
]
# TIP: Utilisez `tf.GradientTape` pour obtenir le gradient de la prédiction par rapport à l'image d'entrée.

# Fonction de preprocessing des images
def preprocess_image(img_path):
    # TODO: Charger et pré-traiter l'image de la même manière que pour le générateur
    img = image.load_img(img_path, target_size=(128, 128))
    img_array = image.img_to_array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

# Fonction pour calucler saliency map
def compute_saliency_map(model, img_array):
    # TODO: Implement saliency map computation using tf.GradientTape
    img_tensor = tf.convert_to_tensor(img_array)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        preds = model(img_tensor, training=False)
        pred_index = tf.argmax(preds[0])
        pred_score = preds[:, pred_index]

    grads = tape.gradient(pred_score, img_tensor)
    saliency = tf.reduce_max(tf.abs(grads), axis=-1)[0]
    saliency = saliency.numpy()

    if np.max(saliency) > 0:
        saliency = saliency / np.max(saliency)

    return saliency


# Function to overlay saliency map on the original image
def overlay_saliency_on_image(img_path, saliency_map):
    # TODO: Charger l'image originale, normaliser la saliency map et superposer les deux pour créer une image d'overlay.
    original = image.load_img(img_path, target_size=(128, 128))
    original_array = image.img_to_array(original) / 255.0

    # Use plt.cm to avoid name collision with confusion matrix variable cm
    colored_saliency = plt.cm.jet(saliency_map)[..., :3]
    overlay = 0.65 * original_array + 0.35 * colored_saliency
    overlay = np.clip(overlay, 0, 1)

    return original_array, overlay
     

# Function to display images with overlays
def display_images_with_overlays(img_paths, saliency_maps):
    # TODO: Afficher les images originales avec les cartes de saillance superposées pour visualiser les régions d'intérêt du modèle.
    n = len(overlay_data)
    if n == 0:
        print('Aucune image valide pour afficher les saliency maps.')
        return

    plt.figure(figsize=(10, 4 * n))
    for i, (img_path, original_img, overlay_img) in enumerate(overlay_data, start=1):
        plt.subplot(n, 2, 2 * i - 1)
        plt.imshow(original_img)
        plt.title(f'Originale\n{os.path.basename(img_path)}')
        plt.axis('off')

        plt.subplot(n, 2, 2 * i)
        plt.imshow(overlay_img)
        plt.title('Saliency Overlay')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


# Process and compute saliency maps for all images
# TODO: Utilisez une boucle pour traiter chaque image, calculer sa carte de saillance et créer une image d'overlay.
overlay_results = []
for path in img_paths:
    if not os.path.exists(path):
        print(f'Image introuvable: {path}')
        continue

    img_batch = preprocess_image(path)
    saliency_map = compute_saliency_map(best_model, img_batch)
    original_img, overlay_img = overlay_saliency_on_image(path, saliency_map)
    overlay_results.append((path, original_img, overlay_img))


# Display images with saliency overlays
#TODO: Afficher les images originales avec les cartes de saillance superposées pour visualiser les régions d'intérêt du modèle.
display_images_with_overlays(img_paths, overlay_results)


# Notes finales

- Expliquez vos résultats et vos observations dans des cellules Markdown.
- Indiquez les améliorations possibles 
- Ajoutez une conclusion courte sur la performance du modèle et les limites du projet.
